# Feature Engineering for Predictive Analytics Model
**Development of a predictive analytics model for predicting sought-after professions**

This notebook demonstrates the complete feature engineering pipeline for transforming
raw vacancy data into ML-ready features.

---

## Table of Contents
1. Setup and Imports
2. Load and Preprocess Data
3. Feature Engineering Pipeline
4. Feature Analysis and Selection
5. Save Processed Features

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Custom modules
from src.preprocessing import VacancyPreprocessor
from src.features import FeatureEngineer, get_top_skills_by_job

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.decomposition import PCA

# Settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

print("✅ Libraries imported!")

## 2. Load and Preprocess Data

In [ ]:
# Initialize preprocessor
preprocessor = VacancyPreprocessor()

# Load data
df = preprocessor.load_data('../results.csv')

# Preprocess
df_processed = preprocessor.preprocess()

print(f"\nDataset shape: {df_processed.shape}")
df_processed.head(3)

## 3. Feature Engineering Pipeline

In [ ]:
# Initialize feature engineer
fe = FeatureEngineer()

# Full pipeline
X, y, feature_columns = fe.prepare_for_modeling(
    df_processed,
    target_column='Job',
    use_tfidf=True,
    use_skills=True,
    tfidf_max_features=50
)

print(f"\nFeature columns ({len(feature_columns)}):")
print(feature_columns[:20], '...')

### 3.1 Analyze Created Features

In [ ]:
# Feature statistics
print("=== Feature Statistics ===")
print(X.describe().round(3))

### 3.2 Top Skills by Profession

In [ ]:
# Get top skills for each profession
top_skills = get_top_skills_by_job(df_processed, top_n=5)

print("=== Top Skills by Profession ===")
for job, skills in list(top_skills.items())[:10]:
    print(f"{job}: {', '.join(skills)}")

### 3.3 Visualize Skills Distribution

In [ ]:
# Count skills across all vacancies
all_skills = []
for skills in df_processed.get('skills_extracted', []):
    if skills:
        all_skills.extend(skills)

# Top 20 skills
skill_counts = Counter(all_skills).most_common(20)
skills_df = pd.DataFrame(skill_counts, columns=['Skill', 'Count'])

plt.figure(figsize=(12, 6))
sns.barplot(data=skills_df, x='Count', y='Skill', palette='viridis')
plt.title('Top-20 Most In-Demand IT Skills', fontsize=14)
plt.xlabel('Frequency')
plt.tight_layout()
plt.show()

## 4. Feature Selection

In [ ]:
# Select top features using ANOVA F-test
selector = SelectKBest(score_func=f_classif, k=30)
X_selected = selector.fit_transform(X, y)

# Get selected feature names
selected_mask = selector.get_support()
selected_features = [feature_columns[i] for i in range(len(feature_columns)) if selected_mask[i]]
scores = selector.scores_[selected_mask]

# Create DataFrame with scores
feature_scores = pd.DataFrame({
    'Feature': selected_features,
    'Score': scores
}).sort_values('Score', ascending=False)

print("=== Top 30 Features by ANOVA F-test ===")
print(feature_scores.to_string(index=False))

### 4.1 Visualize Feature Importance

In [ ]:
plt.figure(figsize=(12, 8))
sns.barplot(data=feature_scores.head(20), x='Score', y='Feature', palette='rocket')
plt.title('Top-20 Most Important Features (ANOVA F-test)', fontsize=14)
plt.xlabel('F-score')
plt.tight_layout()
plt.show()

### 4.2 Correlation Matrix of Selected Features

In [ ]:
# Correlation matrix
top_features = feature_scores['Feature'].head(15).tolist()
corr_matrix = X[top_features].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, fmt='.2f')
plt.title('Correlation Matrix of Top Features', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Train/Test Split

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Number of classes: {len(np.unique(y))}")

## 6. Save Processed Features

In [ ]:
import os

# Create directory if not exists
os.makedirs('../data/processed', exist_ok=True)

# Save features
X_train_df = pd.DataFrame(X_train, columns=feature_columns)
X_test_df = pd.DataFrame(X_test, columns=feature_columns)

X_train_df['target'] = y_train
X_test_df['target'] = y_test

X_train_df.to_csv('../data/processed/train_features.csv', index=False)
X_test_df.to_csv('../data/processed/test_features.csv', index=False)

# Save full processed dataset
df_processed.to_csv('../data/processed/processed_vacancies.csv', index=False)

print("✅ Features saved to data/processed/")
print(f"   - train_features.csv: {X_train_df.shape}")
print(f"   - test_features.csv: {X_test_df.shape}")
print(f"   - processed_vacancies.csv: {df_processed.shape}")

## 7. Summary

In [ ]:
print("=" * 60)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 60)

print(f"\n📊 Dataset:")
print(f"   • Total samples: {len(X)}")
print(f"   • Total features: {len(feature_columns)}")
print(f"   • Number of classes: {len(np.unique(y))}")

print(f"\n🔧 Feature Types:")
skill_features = [f for f in feature_columns if f.startswith('has_')]
tfidf_features = [f for f in feature_columns if f.startswith('tfidf_')]
encoded_features = [f for f in feature_columns if f.endswith('_encoded')]
scaled_features = [f for f in feature_columns if f.endswith('_scaled')]

print(f"   • Skill features: {len(skill_features)}")
print(f"   • TF-IDF features: {len(tfidf_features)}")
print(f"   • Encoded features: {len(encoded_features)}")
print(f"   • Scaled features: {len(scaled_features)}")

print(f"\n📁 Files saved:")
print(f"   • data/processed/train_features.csv")
print(f"   • data/processed/test_features.csv")
print(f"   • data/processed/processed_vacancies.csv")

print("\n" + "=" * 60)
print("✅ Ready for model training!")
print("=" * 60)